# BitNet Studio — Fine-tune Falcon3-10B QLoRA PT-BR + Tool-calling\n\n⚠️ **Atenção**: Falcon 10B em QLoRA 4-bit consome ~12-14GB VRAM.\nNo Google Colab T4 (16GB), use batch=1 e seq_len=128 para evitar OOM.\n\n**Alternativa recomendada**: Colab Pro (A100 40GB) ou RunPod/Vast.ai (RTX 3090 24GB).\n\n**Dataset**: 162 exemplos de tool-calling em PT-BR

In [ ]:
# 1. Instalar dependências\n!pip install -q transformers peft datasets accelerate bitsandbytes safetensors\nimport torch\nprint(f'GPU: {torch.cuda.get_device_name(0)}')\nprint(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# 2. Clonar repo e carregar dataset\n!git clone https://github.com/peder1981/BitNet.git /content/BitNet\nimport json\nfrom datasets import Dataset\n\nDATASET = '/content/BitNet/bitnet-studio/data/ptbr_tools_train_large.jsonl'\nrows = []\nwith open(DATASET, encoding='utf-8') as f:\n    for line in f:\n        if line.strip():\n            rows.append(json.loads(line))\n\ndef to_text(messages):\n    parts = []\n    for m in messages:\n        if m['role'] == 'system':\n            parts.append(f"<|system|>\n{m['content']}")\n        elif m['role'] == 'user':\n            parts.append(f"<|user|>\n{m['content']}")\n        else:\n            parts.append(f"<|assistant|>\n{m['content']}")\n    return '\n'.join(parts) + '\n<|assistant|>\n'\n\ntexts = [to_text(r['messages']) for r in rows]\nprint(f'Dataset: {len(texts)} exemplos')

In [ ]:
# 3. Configurar modelo QLoRA (otimizado para T4 16GB)\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n\nMODEL = 'tiiuae/Falcon3-10B-Instruct'\nOUTPUT = '/content/f10b-ptbr-tools-qlora'\n\ntok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)\nif tok.pad_token is None:\n    tok.pad_token = tok.eos_token\n\n# QLoRA 4-bit com double quant para economizar VRAM\nmodel = AutoModelForCausalLM.from_pretrained(\n    MODEL,\n    load_in_4bit=True,\n    bnb_4bit_compute_dtype=torch.bfloat16,\n    bnb_4bit_use_double_quant=True,\n    bnb_4bit_quant_type='nf4',\n    device_map='auto',\n    trust_remote_code=True,\n    # Otimizações VRAM\n    max_memory={0: '14GiB'},  # Limitar VRAM usada\n)\n\nmodel = prepare_model_for_kbit_training(model)\n\nlora = LoraConfig(\n    r=8,  # Menor rank para economizar VRAM\n    lora_alpha=16,\n    lora_dropout=0.05,\n    bias='none',\n    task_type='CAUSAL_LM',\n    target_modules=['q_proj','k_proj','v_proj','o_proj'],  # Apenas atenção\n)\nmodel = get_peft_model(model, lora)\nmodel.print_trainable_parameters()

In [ ]:
# 4. Tokenizar e treinar (seq_len curta para T4)\nMAX_SEQ_LEN = 128  # Reduzido para economizar VRAM\n\nds = Dataset.from_dict({'text': texts}).map(\n    lambda b: tok(b['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False),\n    batched=True, remove_columns=['text']\n)\n\nargs = TrainingArguments(\n    output_dir=OUTPUT + '/checkpoints',\n    max_steps=300,\n    per_device_train_batch_size=1,\n    gradient_accumulation_steps=8,  # Maior para simular batch maior\n    learning_rate=2e-4,\n    warmup_steps=30,\n    logging_steps=10,\n    save_strategy='steps', save_steps=50,\n    optim='paged_adamw_8bit',\n    fp16=False, bf16=True,\n    seed=42,\n    report_to=[],\n    gradient_checkpointing=True,  # Economiza VRAM\n)\n\ntrainer = Trainer(\n    model=model, args=args, train_dataset=ds,\n    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),\n)\n\ntrainer.train()\nmodel.save_pretrained(OUTPUT)\ntok.save_pretrained(OUTPUT)\nprint(f'Adapter salvo em: {OUTPUT}')

In [ ]:
# 5. Fazer download do adapter\nfrom google.colab import files\nimport shutil\n\nshutil.make_archive('/content/f10b-ptbr-tools-qlora', 'zip', OUTPUT)\nfiles.download('/content/f10b-ptbr-tools-qlora.zip')